In [13]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [14]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [15]:
print(f"Loaded {len(documents)} documents")

Loaded 72 documents


In [16]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

results = index.search("How does the agentic loop keep calling the model until it stops?")

In [17]:
results = index.search("How does the agentic loop keep calling the model until it stops?")
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [18]:
from rag_helper import AgenticRAG
from rag_helper import RAGBase
from openai import OpenAI

llm_client = OpenAI()


In [19]:
assistant = AgenticRAG(index, llm_client)
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

('It keeps calling the model in a `while True` loop and checks each response for tool calls.\n\n- If the model returns a `function_call`, the code runs the tool, appends the tool output to `messages`, and loops again.\n- If the model returns only a normal `message` and no function calls, `has_function_calls` stays `False`, and the loop `break`s.\n\nSo the stop condition is: **no function calls in the latest response**.', ResponseUsage(input_tokens=7126, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7227))


In [20]:
import importlib
import rag_helper

importlib.reload(rag_helper)

from rag_helper import RAGBase, AgenticRAG

assistant = AgenticRAG(index, llm_client)
answer, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print("Input tokens:", usage.input_tokens)

It keeps a `while True` loop and a `has_function_calls` flag.

Flow:
1. Send the full message history to the model.
2. If the response includes a `function_call`, run the tool and append the tool output to `messages`.
3. Set `has_function_calls = True`.
4. If no function calls came back, break out of the loop.

So it stops when the model returns a response with no function calls.
Input tokens: 7126


In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Original documents: {len(documents)}")
print(f"Total chunks: {len(chunks)}")
print(f"\nExample chunk:")
print(chunks[0])

Original documents: 72
Total chunks: 295

Example chunk:
{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "

In [22]:
import minsearch

chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

chunk_assistant = AgenticRAG(chunk_index, llm_client)
answer, usage = chunk_assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print("Input tokens:", usage.input_tokens)

It keeps calling the model in a `while True` loop and uses a flag, `has_function_calls`, to decide when to stop.

- On each iteration, it calls the model.
- If the model returns any `function_call` items, the code runs those tools, appends the results, and sets `has_function_calls = True`.
- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`.
- Then this line ends the loop:

```python
if has_function_calls == False:
    break
```

So the loop stops when the model responds with no more tool calls.
Input tokens: 2309


In [23]:
print("Token reduction:", 7126 - usage.input_tokens)

Token reduction: 4817


In [24]:
def search(query: str) -> str:
    """Search the course materials for information about a topic.
    
    Args:
        query: The search query to look up in the course materials
    """
    results = chunk_index.search(query, num_results=5)
    
    lines = []
    for doc in results:
        lines.append(f"File: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")
    
    return "\n".join(lines).strip()

In [36]:
from toyaikit.chat import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.tools import Tools
from openai import OpenAI

llm_client = OpenAI()

call_count = 0

def search(query: str) -> str:
    """Search the course materials for information about a topic.
    
    Args:
        query: The search query to look up in the course materials
    """
    global call_count
    call_count += 1
    print(f"Search #{call_count}: {query}")
    
    results = chunk_index.search(query, num_results=5)
    lines = []
    for doc in results:
        lines.append(f"File: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()

AGENT_INSTRUCTIONS = """You're a course teaching assistant. Answer the student's question 
using the search tool. Make multiple searches with different keywords before answering."""

tools = Tools()
tools.add_tool(search)

llm = OpenAIClient(model="gpt-4o-mini", client=llm_client)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=AGENT_INSTRUCTIONS,
    llm_client=llm
)

call_count = 0
result = runner.loop("How does the agentic loop work, and how is it different from plain RAG?")

print(result.last_message)
print(f"\nTotal search calls: {call_count}")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}